In [25]:
import os
import json
from dotenv import load_dotenv
from Week1.scraper import fetch_website_contents, fetch_website_links
from openai import OpenAI
from IPython.display import Markdown, display, update_display

In [26]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if api_key and api_key.startswith("sk-proj-") and len(api_key):
    print("API Key found and looks good so far.")
else:
    print("API KEY not found.")

API Key found and looks good so far.


In [27]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

In [28]:
link_system_prompt = """
You are provided with a list of links found on the webpage.
You are able to decide which of this links would be most relevant to include 
in a brochure about the company, such as about page, or a company page, or 
career/jobs pages.

You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "career page", "url": "https://another.full.url.careers"}    
    ]
}

"""

In [29]:
def get_link_user_prompt(url):

    user_prompt = f"""
Here are the list of links on the website {url}.
Please decide which of these are relevant web links for a website brochure about the 
company, respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, and email links
"""
    
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)

    return user_prompt

In [30]:
print(get_link_user_prompt("https://edwarddonner.com"))


Here are the list of links on the website https://edwarddonner.com.
Please decide which of these are relevant web links for a website brochure about the 
company, respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, and email links
https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/01/04/ai-builder-wit

In [31]:
model = "gpt-5-nano"
openai_client = OpenAI()

def select_relevant_links(url):
    print(f"Fetching links from the url {url}....")

    response = openai_client.chat.completions.create(
        model= model,
        messages = [
            {'role': 'system', 'content': link_system_prompt},
            {'role': 'user', 'content': get_link_user_prompt(url)}
        ],
        response_format= {'type': 'json_object'}
    )

    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Fetched {len(links['links'])} relevant links.")

    return links

In [32]:
select_relevant_links("https://edwarddonner.com")

Fetching links from the url https://edwarddonner.com....
Fetched 4 relevant links.


{'links': [{'type': 'company page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'career page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'product page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'}]}

In [33]:
select_relevant_links("https://huggingface.co")

Fetching links from the url https://huggingface.co....
Fetched 8 relevant links.


{'links': [{'type': 'enterprise page',
   'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'branding page', 'url': 'https://huggingface.co/brand'},
  {'type': 'career page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'LinkedIn company page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'}]}

In [34]:
def fetch_all_contents_and_url(url):

    content = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)

    result = f"## Landing Page: \n\n{content}\n\n ## Relevant links:"

    for link in relevant_links['links']:
        result += f"\n\n### Link: Type: {link['type']}\n"
        result += fetch_website_contents(link['url'])

    return result


In [35]:
fetch_all_contents_and_url("https://huggingface.co")

Fetching links from the url https://huggingface.co....
Fetched 9 relevant links.


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


'## Landing Page: \n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.5-35B-A3B\nUpdated\n5 days ago\n•\n769k\n•\n925\nQwen/Qwen3.5-9B\nUpdated\n3 days ago\n•\n172k\n•\n387\nunsloth/Qwen3.5-35B-A3B-GGUF\nUpdated\nabout 10 hours ago\n•\n674k\n•\n495\nQwen/Qwen3.5-27B\nUpdated\n8 days ago\n•\n407k\n•\n570\nQwen/Qwen3.5-0.8B\nUpdated\n2 days ago\n•\n93.4k\n•\n231\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n327\nOmni Video Factory\n🏆\n327\ntext to video, image to video, video extend\nRunning\non\nZero\nFeatured\n1.81k\nQwen Image Multiple Angles 3D Camera\n🎥\n1.81k\nChange the camera angle of a photo with AI\nRunning\non\nZero\nMCP\n1.07k\nWan2.2 14B Prev

In [36]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant webpages from 
company website and create a short brochure about the company for prospective 
customers, investors, and recruits.
Respond in markdown without code blocks.

Include details of company culture, customers, and career/jobs if you have the 
information.
"""

In [37]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}.
Here are the contents of its landing page and other relevant pages;
Use this information to build a short brochure for this company in markdown 
without code blocks. \n\n

"""

    contents = fetch_all_contents_and_url(url)
    user_prompt += contents

    return user_prompt

In [39]:
def create_brochure(company_name, url):

    response = openai_client.chat.completions.create(
        model= model,
        messages= [
            {'role': 'system', 'content': brochure_system_prompt},
            {'role': 'user', 'content': get_brochure_user_prompt(company_name, url)}
        ]
    )

    result = response.choices[0].message.content
    display(Markdown(result))

In [40]:
create_brochure("HuggingFace", "https://huggingface.co")

Fetching links from the url https://huggingface.co....
Fetched 13 relevant links.


# Hugging Face: The AI Community Building the Future

Hugging Face is the collaboration platform at the heart of the machine learning revolution. The mission: empower the next generation of ML engineers, scientists, and end users to learn, collaborate, and share open-source AI to build an open and ethical AI future together.

## What we do
- A thriving community and platform to create, discover, and collaborate on ML models, datasets, and applications.
- A central hub for open-source ML: search and share 2M+ models, 500k+ datasets, and 1M+ applications.
- A multi-modal ecosystem that covers text, image, video, audio, and even 3D, plus tools to build a portfolio and showcase your work.

## Platform & tools
- Hub: The core hub to host, discover, and experiment with models, datasets, and Spaces (interactive apps).
- Spaces: Hosted, interactive AI apps and demos built by the community.
- Datasets: A growing collection for any ML task, with easy sharing and collaboration.
- Learn hub: Courses and tutorials to level up in LLMs, robotics, diffusion, agents, and more.
- Documentation and community: A rich ecosystem of guides, forums, blogs, and open-source tooling.

Key libraries and capabilities you’ll find in the HF ecosystem:
- Transformers, Diffusers, Tokenizers, Datasets, and more for building and deploying cutting-edge models.
- Inference Endpoints and deployment tooling to productionize models at scale.
- Tools for evaluation, training optimization, and efficient inference (PEFT, Optimum, TEI, and related ecosystems).
- Integration with major cloud providers and deployment options (AWS, Google Cloud, Azure, etc.).

## For customers and teams
- Enterprise-grade platform designed to scale with your organization:
  - Single Sign-On (SSO), regions, audit logs, and robust access controls.
  - Resource groups, token management, analytics, and centralized billing.
  - Advanced compute options, including scalable endpoints and ZeroGPU quotas.
  - Private datasets viewer and private storage to protect sensitive work.
  - Inference providers and organization-wide security policies.
  - Priority support and customized contracts to fit your needs.
- Flexible plans:
  - Team plans start at $20 per user per month.
  - Enterprise options available with tailored SLAs and features.

Inference Endpoints (a core production capability):
- One-click deployment from the HF catalog or your own models.
- Fully managed infrastructure with autoscaling for variable traffic.
- Observability through logs and metrics to monitor performance.
- Choice of inference engines (vLLM, TEI, SGLang, etc.) and seamless HF Hub integration.
- Pay-as-you-go pricing for flexible use, with enterprise billing and support options.

## For developers, researchers, and students
- Open-source-first culture: the HF Hub is where you share, explore, and collaborate on ML work.
- Rich learning resources: hands-on courses on LLMs, RL, diffusion, robotics, and more.
- Access to powerful libraries and tooling to build from research to production.
- A community that values openness and ethical AI, with opportunities to contribute, learn, and grow.

## Company culture and community
- The AI community building the future: collaboration, openness, and shared learning.
- A mission to enable open, ethical AI through open-source tooling and collective progress.
- A vibrant ecosystem of blogs, forums, events, and learning resources that invite participation from developers, researchers, and organizations alike.
- Brand emphasis on empowering people to share their work and advance ML responsibly.

## Careers at Hugging Face
- Current openings are posted on the Hugging Face Careers page. If you’re excited by open-source ML, community-driven development, and building tools that accelerate AI for everyone, this is a place to explore.

## Brand and assets
- Hugging Face positions itself as the central hub for open-source ML, powered by a growing community and a stack that enables rapid experimentation and production at scale.
- Core mission: build an open and ethical AI future together, with a strong emphasis on education, collaboration, and responsible AI.

## Get involved
- Explore models, datasets, and Spaces on the platform.
- Sign up to join the community, access learning resources, and start building.
- If you’re representing an organization, explore Enterprise and Team plans to power AI initiatives at scale.
- Learn more through the Learning Hub, Documentation, and Community Forums.

Links to explore (mentioned pages):
- The Landing / Home: Hugging Face – The AI community building the future
- Models, Datasets, Spaces, Community, Docs, Enterprise, Pricing
- Inference Endpoints (production deployment)
- Enterprise: Team & Enterprise Plans
- Careers: Hugging Face – Current Openings
- Learn: LLM Course, MEC/Robotics courses, Diffusion, and more
- Blog and Community Forum

If you’d like, I can tailor this brochure further (e.g., add a one-page executive summary, a version focused on investments, or a version for student recruitment) or format it for print.

In [49]:
def summary(company_name, url):

    response = openai_client.chat.completions.create(
        model= "gpt-4.1-mini",
        messages= [
            {'role': 'system', 'content': brochure_system_prompt},
            {'role': 'user', 'content': get_brochure_user_prompt(company_name, url)}
        ],
        stream=True
    )

    result = ""
    display_handle = display(Markdown(""), display_id=True)

    for chunk in response:
        result += chunk.choices[0].delta.content or ''
        update_display(Markdown(result), display_id= display_handle.display_id)



In [50]:
summary("HuggingFace", "https://huggingface.co")

Fetching links from the url https://huggingface.co....
Fetched 9 relevant links.


# Hugging Face Brochure

## About Hugging Face

Hugging Face is a vibrant AI community and platform committed to building the future of machine learning. It serves as a collaborative hub where machine learning engineers, scientists, and enthusiasts come together to create, explore, and share models, datasets, and AI applications. Hugging Face empowers the next generation of AI innovators through open-source contributions and cutting-edge tools, playing a central role in the AI revolution.

## What They Offer

- **Model Hub:** Access and collaborate on over 2 million machine learning models across numerous modalities including text, image, video, audio, and 3D. Popular trending models include the powerful Qwen series.
- **Datasets:** Explore or contribute to a repository of more than 500,000 datasets, facilitating research and development in diverse AI fields.
- **Spaces:** Host and discover AI demos and applications built by the community, featuring innovative projects like text-to-video generation and AI-driven 3D camera imaging.
- **Enterprise Solutions:** Deploy AI models seamlessly with fully managed Inference Endpoints allowing users to focus on innovation without infrastructure overhead. Features include autoscaling, observability, and integration with the Hugging Face Hub.
- **Open-Source Stack:** Utilize a comprehensive open-source ecosystem to accelerate machine learning development and innovation.

## Company Culture

Hugging Face fosters openness, collaboration, and ethical AI development. The company thrives on community engagement, providing a platform where sharing and discovering new machine learning work is encouraged and celebrated. With a talented science and engineering team pushing technology boundaries, Hugging Face is forging an inclusive and future-oriented AI landscape.

### Core Values

- **Community-Centric:** Encourages active participation in building and sharing AI solutions globally.
- **Transparency:** Open-source philosophy underpins all projects, ensuring accessible and ethical AI.
- **Innovation:** Constantly exploring advanced AI research and real-world applications.
- **Support & Learning:** Provides extensive documentation, forums, and resources to nurture AI talent at all levels.

## Customers and Users

Hugging Face serves a broad spectrum of users, including:

- Individual machine learning practitioners and researchers.
- AI developers building applications across text, image, audio, and 3D.
- Enterprises looking for scalable, secure AI deployment solutions.
- Educational institutions and AI enthusiasts seeking reliable resources and collaboration opportunities.

Major teams and enterprises leverage Hugging Face's Inference Endpoints to run models efficiently at scale.

## Career Opportunities

Hugging Face actively hires passionate and skilled individuals to join their mission of democratizing AI. Current openings span roles in AI research, engineering, product development, and community management. Employees enjoy working at the forefront of AI technology within a supportive, community-driven environment.

- **Visit their Careers page** to discover current job openings.
- Opportunity to contribute to revolutionary open-source machine learning projects.
- Collaborative, innovative, and inclusive culture focused on ethical AI development.

## Connect with Hugging Face

- **Website:** [huggingface.co](https://huggingface.co)
- **Community Forum:** Active discussions and support from AI practitioners worldwide.
- **GitHub:** Open-source repositories and developer tools.
- **Social Media:** Engage with latest news and community events.

---

Join Hugging Face to explore, create, and shape the future of AI together! Whether you are a researcher, developer, or business, Hugging Face provides the tools and community to help you succeed in the rapidly evolving machine learning landscape.